# Fabric IQ Lab – data laden (één notebook)
Dit notebook doet alles in één keer. Je hoeft niets aan de code te veranderen.

1. Koppel links je **Lakehouse** aan dit notebook.
2. Pas eventueel de instellingen in de eerste codecel aan (standaard: Alphen aan den Rijn).
3. Klik op **Run all**. Duurt een paar minuten.

**Wat er gebeurt**
- **A. Fictieve data**: meldingen, werkorders, aannemers en contracten (vaste seed, dus iedereen krijgt hetzelfde).
- **B. Echte open data**: wijken en inwoners (CBS), wijkgrenzen (PDOK), lantaarnpalen (BGT), optioneel kaartlagen van de gemeente.
- **C. Bruikbaar maken**: echt en fictief samenvoegen, unieke sleutels afdwingen, en zes tabellen in de lakehouse zetten: `wijk`, `object`, `melding`, `werkorder`, `aannemer`, `contract`.

Elke tabel heeft een eigen herkomstkolom (`wijk_bron`, `melding_bron`, …) met *Echt – …* of *Fictief*. Lukt een openbare bron niet, dan valt het notebook terug op fictieve data. De lab werkt dus altijd.


In [ ]:
# ===== STAP 1 – PAS DEZE INSTELLINGEN AAN VOOR JOUW GEMEENTE =====
GEMEENTE_NAAM = "Alphen aan den Rijn"
GEMEENTE_CODE = "GM0484"      # CBS-gemeentecode (GM + 4 cijfers)
PEILDATUM     = ""            # leeg = vandaag; of een vaste datum (jjjj-mm-dd)
AANTAL_MELDINGEN = 1600       # fictieve meldingen over ca. 11 maanden
# Optioneel: extra openbare kaartlagen (ArcGIS FeatureServer) van de gemeente. Laat leeg voor andere gemeenten.
EXTRA_KAARTLAGEN = [
    {"objecttype": "Boom",     "url": "https://services6.arcgis.com/J40kr5qRnsaTvT7v/ArcGIS/rest/services/Bomen/FeatureServer/0"},
    {"objecttype": "Afvalbak", "url": "https://services6.arcgis.com/J40kr5qRnsaTvT7v/ArcGIS/rest/services/Afval_bakken/FeatureServer/0"},
]

In [ ]:
# ---- Gedeelde instellingen en hulpfuncties (staat in elk notebook) ----
import json, os, csv, random, math, datetime as dt, urllib.request, urllib.parse
from pathlib import Path

# Map in de lakehouse waar de kit zijn bestanden schrijft (Files/fabric-iq-kit)
KIT = Path(os.environ.get("KIT_PAD", "/lakehouse/default/Files/fabric-iq-kit"))
KIT.mkdir(parents=True, exist_ok=True)
CBS = "https://datasets.cbs.nl/odata/v1/CBS/86165NED"            # Kerncijfers wijken en buurten 2025
PDOK_WIJKEN = "https://api.pdok.nl/cbs/wijken-en-buurten-2025/ogc/v1/collections/wijken/items"
BGT_PAAL = "https://api.pdok.nl/lv/bgt/ogc/v1/collections/paal/items"
KOLOMMEN = {
    "wijk":      ["wijk_id", "naam", "inwoners", "inwoners_per_km2", "woningvoorraad", "bron"],
    "object":    ["object_id", "wijk_id", "objecttype", "conditie", "installatiejaar", "lon", "lat", "bron"],
    "melding":   ["melding_id", "object_id", "categorie", "categoriegroep", "meldingsdatum", "kanaal", "status", "bron"],
    "werkorder": ["werkorder_id", "melding_id", "aannemer_id", "contract_id", "startdatum", "einddatum", "status", "kosten_eur", "bron"],
    "aannemer":  ["aannemer_id", "naam", "specialisme", "bron"],
    "contract":  ["contract_id", "aannemer_id", "startdatum", "einddatum", "sla_dagen", "jaarbudget_eur", "bron"],
}

def get_json(url, params=None, timeout=90):
    if params:
        url += ("&" if "?" in url else "?") + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "fabric-iq-kit"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def schrijf_csv(pad, rijen, kolommen):
    pad = Path(pad); pad.parent.mkdir(parents=True, exist_ok=True)
    with open(pad, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=kolommen, extrasaction="ignore"); w.writeheader(); w.writerows(rijen)

def lees_csv(pad):
    pad = Path(pad)
    if not pad.exists(): return []
    with open(pad, encoding="utf-8") as f: return list(csv.DictReader(f))

def lees_config():
    return json.loads((KIT / "config.json").read_text(encoding="utf-8"))

def veld(props, *delen):
    """Zoek een veld waarvan de naam alle delen bevat (veldnamen verschillen per bron/versie)."""
    for k in props:
        if all(d in k.lower() for d in delen): return props[k]
    return None

## A. Fictieve data maken

In [ ]:
# Instellingen opslaan, zodat stap 2 en 3 ze gebruiken
config = {"gemeente_naam": GEMEENTE_NAAM, "gemeente_code": GEMEENTE_CODE.upper(),
          "bgt_bronhouder": "G" + GEMEENTE_CODE.upper()[2:], "peildatum": PEILDATUM or dt.date.today().isoformat(),
          "aantal_meldingen": AANTAL_MELDINGEN, "extra_kaartlagen": EXTRA_KAARTLAGEN}
(KIT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print("Instellingen opgeslagen in", KIT / "config.json")

In [ ]:
# Wijken: echte wijkcodes en -namen van CBS als die bereikbaar zijn, anders 12 voorbeeldwijken
def cbs_wijknamen(code):
    rijen, d = [], get_json(f"{CBS}/WijkenEnBuurtenCodes", {"$filter": f"startswith(Identifier,'WK{code[2:]}')"})
    while True:
        rijen += d.get("value", [])
        if not d.get("@odata.nextLink"): break
        d = get_json(d["@odata.nextLink"])
    return [(r["Identifier"].strip(), r["Title"].strip()) for r in rijen]
try:
    wijken = cbs_wijknamen(config["gemeente_code"])
    print(f"{len(wijken)} echte wijken van CBS gebruikt")
except Exception as e:
    wijken = []
    print("CBS niet bereikbaar, voorbeeldwijken gebruikt:", str(e)[:100])
if not wijken:
    wijken = [(f"DUMMY{i:02d}", f"Wijk {i:02d}") for i in range(1, 13)]

In [ ]:
# Fictieve data maken (vaste seed = elke keer dezelfde uitkomst)
random.seed(42)
peil = dt.date.fromisoformat(config["peildatum"]); start = peil - dt.timedelta(days=320)
GROEP = {"Boom": "Groen", "Lantaarnpaal": "Straatverlichting", "Wegdek": "Wegen en riolering", "Put": "Wegen en riolering",
         "Bankje": "Afval en meubilair", "Afvalbak": "Afval en meubilair", "Speeltoestel": "Speelvoorzieningen"}
CATEGORIE = {"Groen": ["Boom beschadigd", "Overhangende takken"], "Straatverlichting": ["Lamp defect", "Lantaarnpaal scheef"],
             "Wegen en riolering": ["Gat in wegdek", "Put verstopt"], "Afval en meubilair": ["Afvalbak vol", "Bankje kapot"],
             "Speelvoorzieningen": ["Speeltoestel onveilig"]}
AANNEMERS = [("A01", "Groenwerk Demo BV", "Groen"), ("A02", "Lichtlijn Demo BV", "Straatverlichting"),
             ("A03", "Wegwerk Demo BV", "Wegen en riolering"), ("A04", "Schoon & Heel Demo BV", "Afval en meubilair"),
             ("A05", "SpeelVeilig Demo BV", "Speelvoorzieningen"), ("A06", "Lichtpunt Demo BV", "Straatverlichting")]
SLA = {"Groen": 10, "Straatverlichting": 5, "Wegen en riolering": 7, "Afval en meubilair": 3, "Speelvoorzieningen": 5}

d_wijk = [{"wijk_id": c, "naam": n, "bron": "Fictief"} for c, n in wijken]
d_aannemer = [{"aannemer_id": a, "naam": n, "specialisme": s, "bron": "Fictief"} for a, n, s in AANNEMERS]
d_contract = [{"contract_id": f"C{i:02d}", "aannemer_id": a, "startdatum": f"{peil.year-1}-01-01", "einddatum": f"{peil.year+1}-12-31",
               "sla_dagen": SLA[s], "jaarbudget_eur": random.choice([180000, 240000, 320000, 410000]), "bron": "Fictief"}
              for i, (a, n, s) in enumerate(AANNEMERS, 1)]
d_object, n = [], 1
for c, _ in wijken:
    for t in GROEP:
        for _ in range(random.randint(3, 6)):
            d_object.append({"object_id": f"O{n:05d}", "wijk_id": c, "objecttype": t,
                             "conditie": random.choices(["Goed", "Redelijk", "Matig", "Slecht"], [40, 35, 18, 7])[0],
                             "installatiejaar": random.randint(1985, peil.year - 1), "bron": "Fictief"}); n += 1

d_melding, d_werkorder = [], []
def maak_melding(obj, datum, kanaal=None):
    groep = GROEP[obj["objecttype"]]; mid = f"M{len(d_melding)+1:05d}"
    open_ = (peil - datum).days < random.randint(3, 40)
    d_melding.append({"melding_id": mid, "object_id": obj["object_id"], "categorie": random.choice(CATEGORIE[groep]),
                      "categoriegroep": groep, "meldingsdatum": datum.isoformat(),
                      "kanaal": kanaal or random.choice(["App", "Website", "Telefoon", "Buitendienst"]),
                      "status": "Open" if open_ else "Afgehandeld", "bron": "Fictief"})
    if random.random() < 0.75 or not open_:
        ann = random.choice([a for a in AANNEMERS if a[2] == groep])
        klaar = datum + dt.timedelta(days=max(1, int(random.gauss(6, 3) * (1.9 if ann[0] == "A06" else 1))))
        gereed = klaar <= peil and not open_
        d_werkorder.append({"werkorder_id": f"WO{len(d_werkorder)+1:05d}", "melding_id": mid, "aannemer_id": ann[0],
                            "contract_id": next(c["contract_id"] for c in d_contract if c["aannemer_id"] == ann[0]),
                            "startdatum": (datum + dt.timedelta(days=1)).isoformat(), "einddatum": klaar.isoformat() if gereed else "",
                            "status": "Gereed" if gereed else "In uitvoering", "kosten_eur": round(random.uniform(80, 1400), 2), "bron": "Fictief"})
for _ in range(AANTAL_MELDINGEN):
    maak_melding(random.choice(d_object), start + dt.timedelta(days=random.randint(0, 320)))
# Ingebouwde demo-signalen: herhaalmeldingen op lantaarnpalen en een piek straatverlichting in één wijk
lantaarns = [o for o in d_object if o["objecttype"] == "Lantaarnpaal"]
for _ in range(AANTAL_MELDINGEN // 18):
    maak_melding(random.choice(lantaarns), start + dt.timedelta(days=random.randint(0, 320)))
piek_wijk = wijken[1 if len(wijken) > 1 else 0]
for _ in range(38):
    maak_melding(random.choice([o for o in lantaarns if o["wijk_id"] == piek_wijk[0]]), peil - dt.timedelta(days=random.randint(0, 9)), "App")
d_melding.sort(key=lambda r: r["meldingsdatum"])

for naam, rijen in {"wijk": d_wijk, "object": d_object, "melding": d_melding, "werkorder": d_werkorder,
                    "aannemer": d_aannemer, "contract": d_contract}.items():
    schrijf_csv(KIT / "1-dummy" / f"{naam}.csv", rijen, KOLOMMEN[naam])
config["piek_wijk"] = {"wijk_id": piek_wijk[0], "naam": piek_wijk[1], "categoriegroep": "Straatverlichting"}
(KIT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Fictieve data in {KIT/'1-dummy'}: {len(d_wijk)} wijken, {len(d_object)} objecten, "
      f"{len(d_melding)} meldingen, {len(d_werkorder)} werkorders. Piek ingebouwd in: {piek_wijk[1]}")

## B. Echte open data ophalen (geen inlog nodig)

In [ ]:
# STAP 2 – Openbare bronnen ophalen (geen inlog nodig). Draai eerst stap 1.
config = lees_config(); CODE = config["gemeente_code"]; status = []
UIT = KIT / "2-publiek"; UIT.mkdir(parents=True, exist_ok=True)
MAX_PAGINAS = 300   # veiligheidsgrens bij het ophalen (1000 records per pagina)

def ogc_alles(url, params):
    feats, d = [], get_json(url, {**params, "f": "json", "limit": 1000})
    for _ in range(MAX_PAGINAS):
        feats += d.get("features", [])
        nxt = next((l["href"] for l in d.get("links", []) if l.get("rel") == "next"), None)
        if not nxt: break
        d = get_json(nxt)
    return feats

In [ ]:
# 2a. Wijkgrenzen (PDOK / CBS)
try:
    try:
        feats = ogc_alles(PDOK_WIJKEN, {"gemeentecode": CODE})
        feats = [f for f in feats if str(veld(f["properties"], "gemeentecode") or "").upper() == CODE]
    except Exception:
        feats = []
    if not feats:   # filter niet ondersteund: alles ophalen en zelf filteren
        feats = [f for f in ogc_alles(PDOK_WIJKEN, {}) if str(veld(f["properties"], "gemeentecode") or "").upper() == CODE]
    grenzen = {"type": "FeatureCollection", "features": [
        {"type": "Feature", "geometry": f["geometry"],
         "properties": {"wijk_id": str(veld(f["properties"], "wijkcode")).strip(), "naam": veld(f["properties"], "wijknaam")}}
        for f in feats]}
    (UIT / "wijkgrenzen.geojson").write_text(json.dumps(grenzen), encoding="utf-8")
    status.append(("Wijkgrenzen", "PDOK / CBS", "OK", len(feats)))
except Exception as e:
    status.append(("Wijkgrenzen", "PDOK / CBS", f"FOUT: {str(e)[:80]}", 0))

In [ ]:
# 2b. Kerncijfers per wijk (CBS 86165NED, CC-BY 4.0)
try:
    codes = get_json(f"{CBS}/MeasureCodes")["value"]
    gewenst = {"inwoners": "aantal inwoners", "inwoners_per_km2": "bevolkingsdichtheid", "woningvoorraad": "woningvoorraad"}
    meting = {k: next((c["Identifier"] for c in codes if c["Title"].lower().startswith(v)), None) for k, v in gewenst.items()}
    filt = f"startswith(WijkenEnBuurten,'WK{CODE[2:]}')"
    obs, d = [], get_json(f"{CBS}/Observations", {"$filter": filt})
    while True:
        obs += d.get("value", [])
        if not d.get("@odata.nextLink"): break
        d = get_json(d["@odata.nextLink"])
    namen = {}
    d = get_json(f"{CBS}/WijkenEnBuurtenCodes", {"$filter": f"startswith(Identifier,'WK{CODE[2:]}')"})
    namen = {r["Identifier"].strip(): r["Title"].strip() for r in d.get("value", [])}
    wijk = {w: {"wijk_id": w, "naam": n, "bron": "Echt – CBS"} for w, n in namen.items()}
    for o in obs:
        w = o["WijkenEnBuurten"].strip()
        for k, m in meting.items():
            if m and o["Measure"] == m and w in wijk: wijk[w][k] = o.get("Value")
    schrijf_csv(UIT / "wijk.csv", wijk.values(), KOLOMMEN["wijk"])
    status.append(("Wijken + kerncijfers", "CBS", "OK", len(wijk)))
except Exception as e:
    status.append(("Wijken + kerncijfers", "CBS", f"FOUT: {str(e)[:80]}", 0))

In [ ]:
# 2c. Lantaarnpalen uit de BGT (PDOK / Kadaster, CC0) binnen de gemeente
try:
    gr = json.loads((UIT / "wijkgrenzen.geojson").read_text(encoding="utf-8"))
    def punten(g):
        c = g["coordinates"]; rings = c if g["type"] == "Polygon" else [r for p in c for r in p]
        return [pt for r in rings for pt in r]
    alle = [pt for f in gr["features"] for pt in punten(f["geometry"])]
    bbox = ",".join(str(round(v, 5)) for v in (min(p[0] for p in alle), min(p[1] for p in alle), max(p[0] for p in alle), max(p[1] for p in alle)))
    palen = ogc_alles(BGT_PAAL, {"bbox": bbox})
    rijen = []
    for f in palen:
        p = f["properties"]
        if (str(veld(p, "plus", "type") or "").lower() == "lichtmast" and str(veld(p, "bronhouder") or "").upper() == config["bgt_bronhouder"]
                and not veld(p, "eind", "registratie") and f.get("geometry", {}).get("type") == "Point"):
            lon, lat = f["geometry"]["coordinates"][:2]
            rijen.append({"object_id": f"BGT-{veld(p, 'lokaal', 'id') or f.get('id')}", "objecttype": "Lantaarnpaal",
                          "lon": lon, "lat": lat, "bron": "Echt – PDOK BGT"})
    # BGT bevat historie: houd per object_id alleen de laatste versie over
    uniek = {}
    for r in rijen: uniek[r["object_id"]] = r
    rijen = list(uniek.values())
    schrijf_csv(UIT / "object_bgt_lantaarnpaal.csv", rijen, KOLOMMEN["object"])
    status.append(("Lantaarnpalen", "PDOK BGT", "OK" if rijen else "OK, 0 lichtmasten (houdt de gemeente ze in de BGT bij?)", len(rijen)))
except Exception as e:
    status.append(("Lantaarnpalen", "PDOK BGT", f"FOUT: {str(e)[:80]}", 0))

In [ ]:
# 2d. Extra kaartlagen van de gemeente (ArcGIS), uit de instellingen van stap 1
for laag in config.get("extra_kaartlagen", []):
    try:
        rijen, offset = [], 0
        while True:
            d = get_json(laag["url"] + "/query", {"where": "1=1", "outFields": "OBJECTID", "outSR": 4326, "f": "json",
                                                 "resultOffset": offset, "resultRecordCount": 2000})
            fs = d.get("features", [])
            rijen += [{"object_id": f"{laag['objecttype']}-{veld(f['attributes'], 'objectid') or f'r{offset+i}'}",
                       "objecttype": laag["objecttype"], "lon": f["geometry"]["x"], "lat": f["geometry"]["y"],
                       "bron": "Echt – kaartlaag gemeente"} for i, f in enumerate(fs) if "x" in f.get("geometry", {})]
            if not fs or not d.get("exceededTransferLimit"): break
            offset += len(fs)
        rijen = list({r["object_id"]: r for r in rijen}.values())
        schrijf_csv(UIT / f"object_kaartlaag_{laag['objecttype'].lower()}.csv", rijen, KOLOMMEN["object"])
        status.append((laag["objecttype"], "Kaartlaag gemeente", "OK", len(rijen)))
    except Exception as e:
        status.append((laag["objecttype"], "Kaartlaag gemeente", f"FOUT: {str(e)[:80]}", 0))

In [ ]:
(UIT / "_status.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{'Gegeven':24} {'Bron':20} {'Aantal':>8}  Status")
for g, b, s, n in status: print(f"{g:24} {b:20} {n:>8}  {s}")

## C. Data bruikbaar maken en in de lakehouse zetten

In [ ]:
# STAP 3 – Data bruikbaar maken: echt + fictief samenvoegen tot één schoon model
config = lees_config(); peil = dt.date.fromisoformat(config["peildatum"]); random.seed(7)
D, P, UIT = KIT / "1-dummy", KIT / "2-publiek", KIT / "3-bruikbaar"
dummy = {t: lees_csv(D / f"{t}.csv") for t in KOLOMMEN}
assert dummy["melding"], "Draai eerst stap 1 (fictieve data)."

In [ ]:
# 3a. Wijken: echt (CBS) als beschikbaar, anders fictief
echt_wijk = lees_csv(P / "wijk.csv")
wijk = echt_wijk or dummy["wijk"]
ids = [w["wijk_id"] for w in wijk]
koppel_wijk = {w["wijk_id"]: (w["wijk_id"] if w["wijk_id"] in ids else ids[i % len(ids)]) for i, w in enumerate(dummy["wijk"])}

In [ ]:
# 3b. Objecten: echte objecten krijgen hun wijk via de wijkgrenzen; fictieve objecten vullen de rest aan
def in_ring(x, y, ring):
    binnen, j = False, len(ring) - 1
    for i in range(len(ring)):
        xi, yi = ring[i][:2]; xj, yj = ring[j][:2]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi + 1e-15) + xi: binnen = not binnen
        j = i
    return binnen
def in_vlak(x, y, g):
    vlakken = [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
    return any(in_ring(x, y, v[0]) and not any(in_ring(x, y, h) for h in v[1:]) for v in vlakken)
grenzen = json.loads((P / "wijkgrenzen.geojson").read_text(encoding="utf-8"))["features"] if (P / "wijkgrenzen.geojson").exists() else []
def wijk_van(lon, lat):
    for f in grenzen:
        if in_vlak(lon, lat, f["geometry"]): return f["properties"]["wijk_id"]
    return None

echt_obj = []
for pad in sorted(P.glob("object_*.csv")) if grenzen else []:
    for o in lees_csv(pad):
        w = wijk_van(float(o["lon"]), float(o["lat"]))
        if w in ids:
            o.update(wijk_id=w, conditie="Onbekend", installatiejaar=""); echt_obj.append(o)
echt_types = {o["objecttype"] for o in echt_obj}
object_ = echt_obj + [{**o, "wijk_id": koppel_wijk[o["wijk_id"]]} for o in dummy["object"] if o["objecttype"] not in echt_types]
obj_ids = {o["object_id"] for o in object_}

In [ ]:
# 3c. Meldingen (fictief) koppelen aan echte objecten van hetzelfde type in dezelfde wijk
per_type_wijk, per_type = {}, {}
for o in echt_obj:
    per_type_wijk.setdefault((o["objecttype"], o["wijk_id"]), []).append(o["object_id"])
    per_type.setdefault(o["objecttype"], []).append(o["object_id"])
d_obj = {o["object_id"]: o for o in dummy["object"]}
vervang = {}
melding = []
for m in dummy["melding"]:
    oid = m["object_id"]
    if oid not in obj_ids:
        o = d_obj[oid]; sleutel = (o["objecttype"], koppel_wijk[o["wijk_id"]])
        if oid not in vervang:
            kand = per_type_wijk.get(sleutel) or per_type.get(o["objecttype"])
            vervang[oid] = random.choice(kand) if kand else None
        oid = vervang[oid]
    if oid: melding.append({**m, "object_id": oid})
mel_ids = {m["melding_id"] for m in melding}
werkorder = [w for w in dummy["werkorder"] if w["melding_id"] in mel_ids]
tabellen = {"wijk": wijk, "object": object_, "melding": melding, "werkorder": werkorder,
            "aannemer": dummy["aannemer"], "contract": dummy["contract"]}

In [ ]:
# 3c2. Unieke sleutels afdwingen (dubbele rijen weg, lege sleutels weg)
SLEUTEL = {"wijk": "wijk_id", "object": "object_id", "melding": "melding_id", "werkorder": "werkorder_id",
           "aannemer": "aannemer_id", "contract": "contract_id"}
for t, k in SLEUTEL.items():
    gezien, schoon = set(), []
    for r in tabellen[t]:
        r[k] = str(r.get(k) or "").strip()
        if r[k] and r[k] not in gezien:
            gezien.add(r[k]); schoon.append(r)
    if len(schoon) != len(tabellen[t]): print(f"{t}: {len(tabellen[t]) - len(schoon)} dubbele/lege sleutels verwijderd")
    tabellen[t] = schoon
wijk, object_, melding, werkorder = tabellen["wijk"], tabellen["object"], tabellen["melding"], tabellen["werkorder"]
ids = [w["wijk_id"] for w in wijk]; obj_ids = {o["object_id"] for o in object_}
object_ = tabellen["object"] = [o for o in object_ if o["wijk_id"] in set(ids)]; obj_ids = {o["object_id"] for o in object_}
melding = tabellen["melding"] = [m for m in melding if m["object_id"] in obj_ids]; mel_ids = {m["melding_id"] for m in melding}
werkorder = tabellen["werkorder"] = [w for w in werkorder if w["melding_id"] in mel_ids]

In [ ]:
# 3d. Controles: kloppen alle verwijzingen? Hoeveel is echt / fictief?
fk = {"object→wijk": sum(o["wijk_id"] not in set(ids) for o in object_),
      "melding→object": sum(m["object_id"] not in obj_ids for m in melding),
      "werkorder→melding": sum(w["melding_id"] not in mel_ids for w in werkorder),
      "werkorder→aannemer": sum(w["aannemer_id"] not in {a["aannemer_id"] for a in dummy["aannemer"]} for w in werkorder)}
herkomst = {t: {} for t in tabellen}
for t, rijen in tabellen.items():
    for r in rijen: herkomst[t][r["bron"]] = herkomst[t].get(r["bron"], 0) + 1
for t, rijen in tabellen.items(): schrijf_csv(UIT / f"{t}.csv", rijen, KOLOMMEN[t])
(UIT / "_kwaliteit.json").write_text(json.dumps({"ontbrekende_verwijzingen": fk, "herkomst": herkomst}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Ontbrekende verwijzingen (moet overal 0 zijn):", fk)
dubbel = {t: len(r) - len({x[k] for x in r}) for t, (k, r) in ((t, (SLEUTEL[t], tabellen[t])) for t in tabellen)}
print("Dubbele sleutels (moet overal 0 zijn):", dubbel)
assert not any(fk.values()) and not any(dubbel.values()), "Sleutels kloppen niet – stop."
for t, h in herkomst.items(): print(f"  {t:10} " + ", ".join(f"{b}: {n}" for b, n in h.items()))

In [ ]:
# 3e. Verwachte antwoorden om de data agent mee te toetsen
naam = {w["wijk_id"]: w["naam"] for w in wijk}; o_w = {o["object_id"]: o["wijk_id"] for o in object_}
def top3(rijen): 
    t = {}
    for r in rijen: t[naam[o_w[r["object_id"]]]] = t.get(naam[o_w[r["object_id"]]], 0) + 1
    return sorted(t.items(), key=lambda kv: -kv[1])[:3]
open_m = [m for m in melding if m["status"] == "Open"]
recent_sl = [m for m in melding if m["categoriegroep"] == "Straatverlichting" and (peil - dt.date.fromisoformat(m["meldingsdatum"])).days < 14]
m_obj = {m["melding_id"]: m["object_id"] for m in melding}; m_per_obj = {}
for m in melding: m_per_obj.setdefault(m["object_id"], []).append(dt.date.fromisoformat(m["meldingsdatum"]))
herh, tot = {}, {}
for w in werkorder:
    if w["status"] != "Gereed": continue
    e = dt.date.fromisoformat(w["einddatum"]); a = w["aannemer_id"]; tot[a] = tot.get(a, 0) + 1
    if any(e < d <= e + dt.timedelta(days=30) for d in m_per_obj[m_obj[w["melding_id"]]]): herh[a] = herh.get(a, 0) + 1
an = {a["aannemer_id"]: a["naam"] for a in dummy["aannemer"]}
antwoorden = {"peildatum": config["peildatum"], "let_op": "Meldingen en werkorders zijn fictief; wijken en objecten zijn echt waar <tabel>_bron met 'Echt' begint.",
    "open_meldingen_per_wijk_top3": top3(open_m),
    "open_straatverlichting_per_wijk_top3": top3([m for m in open_m if m["categoriegroep"] == "Straatverlichting"]),
    "straatverlichting_laatste_14_dagen_top3": top3(recent_sl),
    "urgente_meldingen_open_langer_dan_14_dagen": sum((peil - dt.date.fromisoformat(m["meldingsdatum"])).days > 14 for m in open_m),
    "herhaalpercentage_per_aannemer": dict(sorted({an[a]: round(100 * herh.get(a, 0) / n, 1) for a, n in tot.items()}.items(), key=lambda kv: -kv[1]))}
(UIT / "verwachte_antwoorden.json").write_text(json.dumps(antwoorden, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(antwoorden, ensure_ascii=False, indent=2))

In [ ]:
# 3f. Tabellen in de lakehouse zetten (Delta), met de juiste datatypes
if "spark" in globals():
    from pyspark.sql import functions as F
    TYPES = {"datum": ["meldingsdatum", "startdatum", "einddatum"], "geheel": ["inwoners", "woningvoorraad", "installatiejaar", "sla_dagen"],
             "decimaal": ["inwoners_per_km2", "kosten_eur", "jaarbudget_eur", "lon", "lat"]}
    for t in tabellen:
        df = spark.read.option("header", True).csv(f"Files/fabric-iq-kit/3-bruikbaar/{t}.csv")
        # Elke tabel krijgt een EIGEN naam voor de bronkolom (wijk_bron, melding_bron, ...).
        # Zo kan Power BI / de ontologie nooit per ongeluk tabellen koppelen op 'bron'.
        df = df.withColumnRenamed("bron", f"{t}_bron")
        for c in df.columns:
            if c in TYPES["datum"]: df = df.withColumn(c, F.to_date(F.when(F.col(c) != "", F.col(c))))
            elif c in TYPES["geheel"]: df = df.withColumn(c, F.col(c).cast("int"))
            elif c in TYPES["decimaal"]: df = df.withColumn(c, F.col(c).cast("double"))
        df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(t)
        print(f"Tabel {t}: {df.count()} rijen")

## Klaar

In [ ]:
# Klaar: overzicht van de tabellen in de lakehouse
if "spark" in globals():
    SLEUTEL = {"wijk": "wijk_id", "object": "object_id", "melding": "melding_id", "werkorder": "werkorder_id",
               "aannemer": "aannemer_id", "contract": "contract_id"}
    print(f"{'tabel':10} {'rijen':>7} {'dubbele sleutels':>17}")
    for t, k in SLEUTEL.items():
        df = spark.table(t); n = df.count()
        print(f"{t:10} {n:>7} {n - df.select(k).distinct().count():>17}")
    print("\nDubbele sleutels moeten overal 0 zijn. Ga nu verder met stap 3 in de handleiding (Meet the data).")